# Neo4j와 LangChain을 활용한 영화 추천 시스템

---

## 1. Neo4J AuraDB 환경 설정

In [1]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [2]:
from langchain_neo4j import Neo4jGraph

# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph( 
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),
    enhanced_schema=True, # 확장 스키마 출력 설정
)

In [3]:
# 테스트 쿼리 실행 
cypher_query = """
MATCH (n:Movie)
RETURN COUNT(n) AS Movie_Count
"""

graph.query(cypher_query)

[{'Movie_Count': 4803}]

---

## 2. Text-to-Cypher 활용

Text-to-Cypher는 자연어 질의를 Neo4j 데이터베이스 쿼리 언어인 Cypher로 변환하여 지식 그래프를 효과적으로 검색할 수 있는 기술입니다. 영화 데이터베이스에서 사용자 질문을 Neo4j Cypher 쿼리로 변환하는 방법을 배우게 됩니다. 

- **학습 목표**:

   - LLM을 활용하여 자연어 질의를 Cypher 쿼리로 변환하는 방법 이해
   - `GraphCypherQAChain`의 사용법 익히기
   - 다양한 영화 데이터 쿼리 패턴 학습
   - 커스텀 프롬프트를 활용한 정교한 쿼리 생성 방법 습득

- **필요 사항**:
   - Neo4j 데이터베이스 (영화 데이터 포함)
   - 필요 패키지: `langchain-neo4j`, `langchain-openai`

### 2.1 영화 데이터베이스 스키마 확인

영화 데이터베이스는 다음과 같은 노드와 관계로 구성되어 있습니다:

- **노드 속성:**
   - **Person {id: STRING, name: STRING}**: 영화 관련 인물 정보
   - **Movie {id: STRING, title: STRING, released: STRING, rating: FLOAT, overview: STRING, runtime: INTEGER, tagline: STRING, content_embedding: LIST}**: 영화 정보
   - **Genre {id: STRING, name: STRING}**: 영화 장르 정보

- **관계:**
   - **(:Person)-[:ACTED_IN]->(:Movie)**: 배우와 영화 간의 출연 관계
   - **(:Person)-[:DIRECTED]->(:Movie)**: 감독과 영화 간의 감독 관계
   - **(:Movie)-[:IN_GENRE]->(:Genre)**: 영화와 장르 간의 분류 관계

In [4]:
graph.refresh_schema()
print(graph.schema)

Node properties:
- **Person**
  - `name`: STRING Example: "D.W. Griffith"
  - `id`: STRING Example: "person-D.W. Griffith"
- **Movie**
  - `rating`: FLOAT Min: 0.0, Max: 10.0
  - `id`: STRING Example: "movie-4594"
  - `title`: STRING Example: "The Broadway Melody"
  - `released`: STRING Example: "1929-02-08"
  - `overview`: STRING Example: "Harriet and Queenie Mahoney, a vaudeville act, com"
  - `runtime`: INTEGER Min: 0, Max: 338
  - `tagline`: STRING Example: "The pulsating drama of Broadway's bared heart spea"
- **Genre**
  - `name`: STRING Example: "Drama"
  - `id`: STRING Example: "genre-Drama"
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


### 2.2 LangChain을 활용한 Text-to-Cypher 구현

- LangChain으로 Neo4J 지식 그래프 조회
- LLM을 사용하여 사용자 질의를 Cypher 쿼리로 변환

#### 1) **GraphCypherQAChain** 설정

In [6]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

# LLM 및 그래프 객체 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

# GraphCypherQAChain 객체 초기화
# 자연어 질의를 Cypher 쿼리로 변환하고 실행하는 체인
# llm: 질의를 Cypher로 변환할 LLM 모델
# graph: 쿼리를 실행할 Neo4j 그래프 객체
# allow_dangerous_requests=True: 잠재적으로 위험한 쿼리도 허용
# verbose=True: 중간 과정과 디버깅 정보를 상세히 출력
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
)

#### 2) **기본 영화 쿼리** 

- Text to Cypher (DB 조회)

In [7]:
# 영화 제목으로 정보 검색
answer = cypher_chain.invoke({"query": "영화 'Apollo 13'에 대한 정보를 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Apollo 13'}) RETURN m.id, m.title, m.released, m.rating, m.overview, m.runtime, m.tagline
Full Context:
[{'m.id': 'movie-635', 'm.title': 'Apollo 13', 'm.released': '1995-06-30', 'm.rating': 7.3, 'm.overview': 'The true story of technical troubles that scuttle the Apollo 13 lunar mission in 1971, risking the lives of astronaut Jim Lovell and his crew, with the failed journey turning into a thrilling saga of heroism. Drifting more than 200,000 miles from Earth, the astronauts work furiously with the ground crew to avert tragedy.', 'm.runtime': 140, 'm.tagline': 'Houston, we have a problem.'}]

> Finished chain.


In [8]:
# 답변 출력
print(answer)

{'query': "영화 'Apollo 13'에 대한 정보를 알려주세요.", 'result': '영화 \'Apollo 13\'은 1995년 6월 30일에 개봉되었으며, 러닝타임은 140분입니다. 이 영화는 1971년 아폴로 13호 달 탐사 임무 중 발생한 기술적 문제로 인해 우주비행사 짐 러벨과 그의 승무원들의 생명이 위태로워진 실제 이야기를 다루고 있습니다. 지구에서 20만 마일 이상 떨어진 우주에서 우주비행사들과 지상 관제팀이 함께 노력하여 비극을 막아내는 긴장감 넘치는 영웅담을 그렸습니다. 영화의 평점은 7.3이며, 유명한 대사인 "Houston, we have a problem."이 태그라인으로 사용되었습니다.'}


In [9]:
# LLM 답변 출력
from pprint import pprint
pprint(answer['result'])

("영화 'Apollo 13'은 1995년 6월 30일에 개봉되었으며, 러닝타임은 140분입니다. 이 영화는 1971년 아폴로 13호 달 "
 '탐사 임무 중 발생한 기술적 문제로 인해 우주비행사 짐 러벨과 그의 승무원들의 생명이 위태로워진 실제 이야기를 다루고 있습니다. 지구에서 '
 '20만 마일 이상 떨어진 우주에서 우주비행사들과 지상 관제팀이 함께 노력하여 비극을 막아내는 긴장감 넘치는 영웅담을 그렸습니다. 영화의 '
 '평점은 7.3이며, 유명한 대사인 "Houston, we have a problem."이 태그라인으로 사용되었습니다.')


In [10]:
# 특정 감독의 영화 찾기
answer = cypher_chain.invoke({"query": "'Christopher Nolan' 감독의 영화를 모두 찾아주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie) RETURN m
Full Context:
[{'m': {'overview': "Suffering short-term memory loss after a head injury, Leonard Shelby embarks on a grim quest to find the lowlife who murdered his wife in this gritty, complex thriller that packs more knots than a hangman's noose. To carry out his plan, Shelby snaps Polaroids of people and places, jotting down contextual notes on the backs of photos to aid in his search and jog his memory. He even tattoos his own body in a desperate bid to remember.", 'rating': 8.1, 'runtime': 113, 'tagline': 'Some memories are best forgotten.', 'content_embedding': [0.00442682346329093, 0.05290191248059273, -0.07258890569210052, 0.01242810022085905, -0.018738385289907455, -0.004265285562723875, 0.04404826462268829, 0.012923024594783783, -0.016511226072907448, -0.02719334326684475, 0.003921587951481342, -0.03794419765472412, -0.049574922770261765,

In [ ]:
# 답변 출력
print(answer)

In [ ]:
# LLM 답변 출력
pprint(answer['result'])

In [ ]:
# 특정 배우가 출연한 영화 찾기
answer = cypher_chain.invoke({"query": "'Tom Hanks'가 출연한, 평점이 가장 높은 영화 3개는 무엇인가요?"})

In [ ]:
# 답변 출력
print(answer)

In [ ]:
# LLM 답변 출력
pprint(answer['result'])

In [ ]:
# 장르별 영화 검색
answer = cypher_chain.invoke({"query": "2010년 이후 개봉한 'Drama' 장르 영화 중 평점이 높은 순서로 5개를 보여주세요."})

In [ ]:
# 답변 출력
print(answer)

In [ ]:
# LLM 답변 출력
pprint(answer['result'])

#### 3) **출력 갯수를 지정 (top k)** 

In [11]:
# top_k 파라미터로 결과 수 제한
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    top_k=5,  # 최대 5개의 결과만 반환
)

answer = cypher_chain.invoke({"query": "'Tom Hanks' 배우가 출연한 영화를 모두 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN m
Full Context:
[{'m': {'overview': "A successful businessman falls in love with the girl of his dreams. There's one big complication though; he's fallen hook, line and sinker for a mermaid.", 'rating': 6.1, 'runtime': 111, 'tagline': "Allen Bauer Thought He'd Never Find The Right Woman... He Was Only Half Wrong!", 'content_embedding': [-0.013971338048577309, 0.02177574299275875, 0.015516372397542, 0.034096404910087585, 0.009171166457235813, -0.024905428290367126, -0.0023967844899743795, 0.010214394889771938, -0.024152720347046852, -0.029685791581869125, -0.01778770610690117, -0.06893230974674225, -0.015661632642149925, -0.02123432047665119, -0.006196645088493824, 0.06676661968231201, 0.00993708148598671, -0.060533661395311356, -0.02130034752190113, 0.029632970690727234, 0.01485610194504261, -0.0077515835873782635, -0.009494699537754059, -0.01412980351597070

In [12]:
# LLM 답변 출력
pprint(answer['result'])

('Tom Hanks 배우가 출연한 영화는 다음과 같습니다: "Splash" (1984), "Big" (1988), '
 '"Philadelphia" (1993), 그리고 "Forrest Gump" (1994)입니다.')


#### 4) **중간 과정 확인** 

- `return_intermediate_steps`=True

In [13]:
# 중간 결과를 포함하여 출력
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_intermediate_steps=True  # 생성된 Cypher 쿼리와 중간 결과 확인
)

# 평점 8점 이상인 액션 영화 찾기
answer = cypher_chain.invoke({"query": "평점이 8점 이상인 'Action' 장르 영화는 무엇이 있나요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Action'})
WHERE m.rating >= 8
RETURN m.title, m.rating
Full Context:
[{'m.title': 'Seven Samurai', 'm.rating': 8.2}, {'m.title': 'Star Wars', 'm.rating': 8.1}, {'m.title': 'The Empire Strikes Back', 'm.rating': 8.2}, {'m.title': 'Scarface', 'm.rating': 8.0}, {'m.title': "One Man's Hero", 'm.rating': 9.3}, {'m.title': 'The Lord of the Rings: The Fellowship of the Ring', 'm.rating': 8.0}, {'m.title': 'The Lord of the Rings: The Two Towers', 'm.rating': 8.0}, {'m.title': 'Oldboy', 'm.rating': 8.0}, {'m.title': 'The Lord of the Rings: The Return of the King', 'm.rating': 8.1}, {'m.title': 'Star Wars: Clone Wars: Volume 1', 'm.rating': 8.0}]

> Finished chain.


In [14]:
# 중간 결과 확인
for k, v in answer.items():
    print(f"{k}: {v}")

query: 평점이 8점 이상인 'Action' 장르 영화는 무엇이 있나요?
result: 평점이 8점 이상인 'Action' 장르 영화로는 다음과 같은 작품들이 있습니다: Seven Samurai (8.2), Star Wars (8.1), The Empire Strikes Back (8.2), Scarface (8.0), The Lord of the Rings: The Fellowship of the Ring (8.0), The Lord of the Rings: The Two Towers (8.0), Oldboy (8.0), The Lord of the Rings: The Return of the King (8.1), Star Wars: Clone Wars: Volume 1 (8.0)입니다.
intermediate_steps: [{'query': "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Action'})\nWHERE m.rating >= 8\nRETURN m.title, m.rating"}, {'context': [{'m.title': 'Seven Samurai', 'm.rating': 8.2}, {'m.title': 'Star Wars', 'm.rating': 8.1}, {'m.title': 'The Empire Strikes Back', 'm.rating': 8.2}, {'m.title': 'Scarface', 'm.rating': 8.0}, {'m.title': "One Man's Hero", 'm.rating': 9.3}, {'m.title': 'The Lord of the Rings: The Fellowship of the Ring', 'm.rating': 8.0}, {'m.title': 'The Lord of the Rings: The Two Towers', 'm.rating': 8.0}, {'m.title': 'Oldboy', 'm.rating': 8.0}, {'m.title': 'The Lord of 

#### 5) **직접 Cypher 결과 얻기 (LLM 답변 없이)** 

- `return_direct`=True

In [15]:
# 직접 Cypher 결과 얻기 (LLM 답변 없이)
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_direct=True  # LLM 답변 생성 단계 건너뛰기
)

# 2000년대 개봉한 영화 중 평점 순 정렬
answer = cypher_chain.invoke({"query": "2000년대(2000-01-01 ~ 2009-12-31) 개봉한 영화를 평점 순으로 정렬해주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie) 
WHERE m.released >= '2000-01-01' AND m.released <= '2009-12-31' 
RETURN m 
ORDER BY m.rating DESC

> Finished chain.


In [16]:
for k, v in answer.items():
    print(f"{k}: {v}")

query: 2000년대(2000-01-01 ~ 2009-12-31) 개봉한 영화를 평점 순으로 정렬해주세요.
result: [{'m': {'overview': 'An aging out of work clown returns to his small hometown, resigned to spend the rest of his days in a drunken stupor. But when his passion for clowning is reawakened by the local amateur circus he finds his smile.', 'rating': 10.0, 'runtime': 0, 'content_embedding': [-0.02406313084065914, 0.03388175740838051, -0.007844008505344391, 0.02865242026746273, -0.045811187475919724, -0.011439179070293903, 0.0420253649353981, 0.007551219779998064, -0.024294637143611908, 0.004630143754184246, 0.013052919879555702, -0.05730484053492546, -0.04867098107933998, 0.049025051295757294, 0.010254406370222569, 0.04093591868877411, 0.0017328994581475854, 0.02718166820704937, -0.016791081055998802, 0.0660204067826271, 0.03039553202688694, -0.014326210133731365, 0.013270809315145016, -0.043741241097450256, 0.038048889487981796, -0.005900029093027115, -0.04744535684585571, 0.006907766219228506, 0.0744636058807373, 0.004

In [ ]:
import pandas as pd
pd.DataFrame([item['m'] for item in answer['result']])

#### 6) **다양한 LLM 모델 활용** 

- cypher 쿼리 생성하는 모델과 최종 답변 생성 모델을 다르게 적용

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Cypher 쿼리 생성은 OpenAI 모델, 최종 답변은 Gemini 모델 사용
cypher_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)
qa_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=cypher_llm,  # Cypher 쿼리 생성용 LLM
    qa_llm=qa_llm,          # 최종 답변 생성용 LLM
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
)

# 특정 배우와 감독이 함께 작업한 영화 찾기
answer = cypher_chain.invoke({"query": "배우 'Leonardo DiCaprio'와 감독 'Christopher Nolan'이 함께 작업한 영화가 있나요?"})

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")

#### 7) **커스텀 프롬프트 활용**

In [17]:
from langchain_core.prompts.prompt import PromptTemplate

# Cypher 생성을 위한 영화 데이터베이스 특화 프롬프트
CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to question a movie graph database.
Instructions:
- Use only the provided node labels, relationship types, and properties in the schema.
- Do not use any relationship types or properties not specified in the schema.
- Focus on extracting meaningful insights from movie data.

Schema:
{schema}

Note: 
- Provide only the Cypher statement.
- Do not include explanations or apologies.
- Generate precise, relevant Cypher queries.

Examples:
# 특정 배우가 출연한 영화 찾기
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
WHERE p.name = 'Tom Hanks'
RETURN m.title, m.released, m.rating
ORDER BY m.released DESC

# 특정 장르의 평점 높은 영화 찾기
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
WHERE g.name = 'Action'
RETURN m.title, m.rating
ORDER BY m.rating DESC
LIMIT 10

# 특정 감독의 영화 중 가장 흥행한 작품
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE p.name = 'Steven Spielberg'
RETURN m.title, m.rating
ORDER BY m.rating DESC
LIMIT 5

# 특정 연도에 개봉한 영화 조회
MATCH (m:Movie)
WHERE m.released = 2000
RETURN m.title, m.rating
ORDER BY m.rating DESC

# 특정 배우와 감독이 함께 작업한 영화 찾기
MATCH (actor:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(director:Person)
WHERE actor.name = 'Leonardo DiCaprio' AND director.name = 'Christopher Nolan'
RETURN m.title, m.released, m.rating

The question is:
{question}"""

# 결과 처리를 위한 영화 QA 프롬프트
QA_TEMPLATE = """
당신은 영화 데이터베이스 분석 전문가로, 영화 정보에 대한 명확하고 간결한 정보를 한국어로 제공합니다.

[질문]
{question}

[검색 결과]
{context}

# 응답 가이드라인:
- 검색 결과에서 핵심 정보를 요약하세요
- 영화 데이터에 대한 명확하고 객관적인 개요를 제공하세요
- 전문적이고 유익한 톤을 사용하세요
- 영화 데이터에서 중요한 패턴이나 트렌드를 강조하세요
- 맥락이 불충분한 경우 더 많은 정보가 필요하다고 명확히 언급하세요
- 추측이나 개인적인 해석은 피하세요

# 응답 형식:
- 간략한 발견 요약으로 시작하세요
- 여러 영화가 발견된 경우 간결한 개요를 제공하세요
- 가독성을 위해 글머리 기호나 짧은 단락을 사용하세요
- 개봉일, 평점, 주요 배우나 감독과 같은 관련 세부 정보를 포함하세요
- 모든 숫자 데이터나 기술 용어를 이해하기 쉬운 언어로 번역하세요

# 예시 응답 구조:
"분석 결과, [주요 발견 요약]

주요 특징:
- [첫 번째 중요 인사이트]
- [두 번째 중요 인사이트]

추가 정보: [필요한 경우 추가 설명]"
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], 
    template=CYPHER_GENERATION_TEMPLATE
)

QA_PROMPT = PromptTemplate(
    input_variables=["question", "context"], 
    template=QA_TEMPLATE
)

# Chain 생성 - 주목: input_key와 output_key를 명시적으로 설정
cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=cypher_llm,
    qa_llm=qa_llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=QA_PROMPT,
    input_key="question",  
    output_key="result"
)

# Cypher 쿼리 실행
answer = cypher_chain.invoke({"question": "배우 'Leonardo DiCaprio'와 감독 'Christopher Nolan'이 함께 작업한 영화가 있나요?"})

NameError: name 'cypher_llm' is not defined

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")